# 🏠 NhaDat Chatbot - Trainer (mọi định dạng)
**Đọc TẤT CẢ file từ Google Drive (PDF, Sheets, Slides, Docs, Word, Excel, PowerPoint, ảnh) → Gemini Vision/parse → Markdown → Push GitHub**

### Làm 2 việc:
1. Vào **Secrets (🔑 bên trái)** → thêm `GEMINI_API_KEY` và `GITHUB_TOKEN`
2. **Runtime → Run all**

Khi chạy Cell 2 lần đầu sẽ hiện popup → chọn account **infonhadat2000@gmail.com** (account được share folder).

In [ ]:
# CELL 1: Cài thư viện (có LibreOffice để convert Office → PDF)
!pip install -q pymupdf pdfplumber Pillow requests google-api-python-client google-auth pandas openpyxl
!apt-get install -y -qq libreoffice > /dev/null 2>&1
print('✅ Xong cài đặt')

In [ ]:
# CELL 2: CẤU HÌNH (không cần sửa)
import os, base64, time, requests, io, subprocess
from pathlib import Path
from google.colab import auth, userdata
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google.auth import default

DRIVE_FOLDER_ID = '18AYPZ30KCRDEm2a7p2TWgHHUKW2r7v-W'
GITHUB_OWNER    = 'quang507'
GITHUB_REPO     = 'NhaDat-chatbot'
GITHUB_BRANCH   = 'main'
WORK_DIR        = '/content/work'
PDF_DIR         = '/content/pdfs'
OUTPUT_DIR      = '/content/output_md'

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
GITHUB_TOKEN   = userdata.get('GITHUB_TOKEN')

for d in (WORK_DIR, PDF_DIR, OUTPUT_DIR): os.makedirs(d, exist_ok=True)

auth.authenticate_user()
creds, _ = default()
drive_service = build('drive', 'v3', credentials=creds)

print(f'🔑 Gemini: {"✅" if GEMINI_API_KEY else "❌ CHƯA SET"}')
print(f'🔑 GitHub: {"✅" if GITHUB_TOKEN else "❌ CHƯA SET"}')
print('✅ Đã kết nối Google Drive')

In [ ]:
# CELL 3: Liệt kê TẤT CẢ file trong folder (đệ quy) + thống kê định dạng
GOOGLE_DOC   = 'application/vnd.google-apps.document'
GOOGLE_SLIDE = 'application/vnd.google-apps.presentation'
GOOGLE_SHEET = 'application/vnd.google-apps.spreadsheet'
FOLDER       = 'application/vnd.google-apps.folder'

def list_all(folder_id, service, prefix=''):
    out = []
    token = None
    while True:
        resp = service.files().list(
            q=f"'{folder_id}' in parents and trashed=false",
            fields='nextPageToken, files(id, name, mimeType)',
            pageToken=token, pageSize=1000,
            supportsAllDrives=True, includeItemsFromAllDrives=True
        ).execute()
        for f in resp.get('files', []):
            f['path'] = prefix + f['name']
            if f['mimeType'] == FOLDER:
                out.extend(list_all(f['id'], service, prefix + f['name'] + '/'))
            else:
                out.append(f)
        token = resp.get('nextPageToken')
        if not token: break
    return out

all_files = list_all(DRIVE_FOLDER_ID, drive_service)
print(f'🔍 Tìm thấy tổng {len(all_files)} file')
from collections import Counter
for mt, n in Counter(f['mimeType'] for f in all_files).most_common():
    print(f'  {n:3d} × {mt}')

In [ ]:
# CELL 4: Tải & convert mọi file → PDF (hoặc giữ ảnh để OCR)
def download_media(file_id, dest):
    req = drive_service.files().get_media(fileId=file_id)
    with open(dest, 'wb') as fh:
        dl = MediaIoBaseDownload(fh, req)
        done = False
        while not done: _, done = dl.next_chunk()

def export_pdf(file_id, dest):
    req = drive_service.files().export_media(fileId=file_id, mimeType='application/pdf')
    with open(dest, 'wb') as fh:
        dl = MediaIoBaseDownload(fh, req)
        done = False
        while not done: _, done = dl.next_chunk()

def export_sheet_csv(file_id, dest):
    req = drive_service.files().export_media(fileId=file_id, mimeType='text/csv')
    data = req.execute()
    Path(dest).write_bytes(data)

def office_to_pdf(src, outdir):
    subprocess.run(['libreoffice','--headless','--convert-to','pdf','--outdir',outdir,src],
                   check=False, capture_output=True, timeout=180)
    cand = os.path.join(outdir, Path(src).stem + '.pdf')
    return cand if os.path.exists(cand) else None

IMG_TYPES = ('image/png','image/jpeg','image/jpg','image/webp')
OFFICE_EXT = ('.docx','.doc','.pptx','.ppt','.xlsx','.xls')

jobs = []  # (kind, path, name)  kind in {pdf, image, csv}
for f in all_files:
    name, mt, fid = f['path'], f['mimeType'], f['id']
    safe = name.replace('/', '__')
    try:
        if mt == 'application/pdf':
            dest = os.path.join(PDF_DIR, safe if safe.lower().endswith('.pdf') else safe+'.pdf')
            download_media(fid, dest); jobs.append(('pdf', dest, name))
        elif mt in (GOOGLE_DOC, GOOGLE_SLIDE):
            dest = os.path.join(PDF_DIR, safe + '.pdf')
            export_pdf(fid, dest); jobs.append(('pdf', dest, name))
        elif mt == GOOGLE_SHEET:
            dest = os.path.join(WORK_DIR, safe + '.csv')
            export_sheet_csv(fid, dest); jobs.append(('csv', dest, name))
        elif mt in IMG_TYPES:
            ext = '.' + mt.split('/')[-1]
            dest = os.path.join(WORK_DIR, safe + ext)
            download_media(fid, dest); jobs.append(('image', dest, name))
        elif name.lower().endswith(OFFICE_EXT):
            raw = os.path.join(WORK_DIR, safe)
            download_media(fid, raw)
            pdf = office_to_pdf(raw, PDF_DIR)
            if pdf: jobs.append(('pdf', pdf, name))
            else: print(f'  ⚠️ Không convert được: {name}')
        elif name.lower().endswith(('.csv','.txt','.md')):
            dest = os.path.join(WORK_DIR, safe)
            download_media(fid, dest); jobs.append(('csv', dest, name))
        else:
            print(f'  ⏭️ Bỏ qua (không hỗ trợ): {name} [{mt}]')
            continue
        print(f'  ✅ {name}')
    except Exception as e:
        print(f'  ❌ {name}: {e}')
print(f'\n📥 Chuẩn bị xử lý {len(jobs)} file')

In [ ]:
# CELL 5: Hàm trích xuất (PDF/ảnh qua Vision, CSV/text đọc thẳng)
import fitz, pdfplumber
GEMINI_URL = f'https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?key={GEMINI_API_KEY}'
PROMPT = '''Bạn là chuyên gia trích xuất dữ liệu bất động sản.
Đọc hình ảnh và chuyển TOÀN BỘ nội dung thành Markdown có cấu trúc.
- Giữ NGUYÊN mọi số liệu: diện tích, giá, mã căn/lô, kích thước, tỉ lệ thanh toán
- Bảng → Markdown table | Tiêu đề → ## | Danh sách → bullet
- Sơ đồ/mặt bằng: liệt kê từng lô/căn với số hiệu và diện tích
- KHÔNG thêm nhận xét, chỉ chuyển dữ liệu. Trả về Markdown thuần.'''

def gemini_ocr(img_bytes, retry=4):
    b64 = base64.b64encode(img_bytes).decode()
    payload = {'contents':[{'parts':[{'text':PROMPT},{'inline_data':{'mime_type':'image/png','data':b64}}]}],
               'generationConfig':{'temperature':0.1,'maxOutputTokens':8192}}
    for i in range(retry):
        r = requests.post(GEMINI_URL, json=payload, timeout=120)
        if r.status_code == 429:
            w=(i+1)*20; print(f'    ⏳ rate limit {w}s'); time.sleep(w); continue
        r.raise_for_status()
        return r.json()['candidates'][0]['content']['parts'][0]['text']
    return ''

def page_png(page, dpi=200):
    return page.get_pixmap(matrix=fitz.Matrix(dpi/72,dpi/72), colorspace=fitz.csRGB).tobytes('png')

def pdf_text_page(path, i):
    with pdfplumber.open(path) as pdf:
        p = pdf.pages[i]; parts=[]
        for tbl in (p.extract_tables() or []):
            rows=[[str(c or '').strip() for c in r] for r in tbl if r]
            if not rows: continue
            h=rows[0]; md='| '+' | '.join(h)+' |\n| '+' | '.join(['---']*len(h))+' |\n'
            for r in rows[1:]:
                while len(r)<len(h): r.append('')
                md+='| '+' | '.join(r)+' |\n'
            parts.append(md)
        t=p.extract_text() or ''
        if t.strip(): parts.append(t.strip())
        return '\n\n'.join(parts)

def do_pdf(path, name):
    doc=fitz.open(path); parts=[f'# {name}']
    for i,page in enumerate(doc):
        print(f'    trang {i+1}/{len(doc)}', end=' ')
        if len(page.get_text().strip())>=80:
            print('(text)',end=' '); txt=pdf_text_page(path,i)
            if txt.strip(): parts.append(f'## Trang {i+1}\n\n{txt.strip()}')
        else:
            print('(vision)',end=' '); md=gemini_ocr(page_png(page))
            if md.strip(): parts.append(f'## Trang {i+1}\n\n{md.strip()}')
            time.sleep(2)
        print('✅')
    doc.close(); return '\n\n---\n\n'.join(parts)

def do_image(path, name):
    print('    (vision ảnh)', end=' ')
    md=gemini_ocr(Path(path).read_bytes()); time.sleep(2); print('✅')
    return f'# {name}\n\n{md.strip()}'

def do_csv(path, name):
    raw=Path(path).read_text(encoding='utf-8', errors='ignore')
    return f'# {name}\n\n```\n{raw.strip()}\n```'

print('✅ Sẵn sàng')

In [ ]:
# CELL 6: Xử lý tất cả → markdown
results = {}
for kind, path, name in jobs:
    print(f'📄 {name}')
    try:
        if kind=='pdf':   md = do_pdf(path, name)
        elif kind=='image': md = do_image(path, name)
        else:             md = do_csv(path, name)
        fn = name.replace('/','__').rsplit('.',1)[0] + '.md'
        results[fn] = md
        Path(os.path.join(OUTPUT_DIR, fn)).write_text(md, encoding='utf-8')
        print(f'  💾 {fn} ({len(md):,} ký tự)')
    except Exception as e:
        print(f'  ❌ {name}: {e}')
print(f'\n✅ Xong {len(results)}/{len(jobs)} file')

In [ ]:
# CELL 7: Push lên GitHub → data/drive-extracted/
def gh_push(repo_path, content, msg):
    url=f'https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}/contents/{repo_path}'
    hdrs={'Authorization':f'Bearer {GITHUB_TOKEN}','Accept':'application/vnd.github+json'}
    r=requests.get(f'{url}?ref={GITHUB_BRANCH}', headers=hdrs)
    sha=r.json().get('sha') if r.ok else None
    body={'message':msg,'branch':GITHUB_BRANCH,'content':base64.b64encode(content.encode()).decode()}
    if sha: body['sha']=sha
    r=requests.put(url, headers=hdrs, json=body)
    print(f'  {"✅" if r.ok else "❌"} {repo_path}' + ('' if r.ok else f' {r.text[:120]}'))
    return r.ok

if not GITHUB_TOKEN:
    print('⚠️ Thiếu GITHUB_TOKEN → file lưu tại /content/output_md/')
elif not results:
    print('⚠️ Không có kết quả để push (results rỗng)')
else:
    print('🚀 Pushing...')
    ok=0
    for fn, content in results.items():
        if gh_push(f'data/drive-extracted/{fn}', content, f'data: {fn} via Colab'): ok+=1
        time.sleep(1)
    print(f'\n✅ {ok}/{len(results)} file → data/drive-extracted/')
    print('🔔 Về máy chạy Chay_Dong_Bo.bat để rebuild RAG index!')